# PPI walkthrough

Position-aware Player Performance Index on FBref match data.

**Pipeline:** weights → MinMax → `w·x̃` → cohort rescale → ranking → linear surrogate → holdout check.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from ppi.io import load_match_stats, load_weights, DATA_RAW
from ppi.scoring import compute_ppi
from ppi.explain import feature_contributions
from ppi.ranking import rank_players
from ppi.scoring import scale_features
from ppi.validation import compare_expert_vs_surrogate
from ppi.models import PositionModelBundle

MATCH_ID = 'el-clasico-2025-05-11'

## 1. Load match + weights

In [ ]:
match = load_match_stats(match_id=MATCH_ID)
weights = load_weights(DATA_RAW / 'importance_weights.csv')
match.head()

## 2. Expert PPI + ranking

In [ ]:
scored, scaler, _ = compute_ppi(match, weights)
ranked = rank_players(scored, min_minutes=30)
ranked

## 3. Feature contributions (Raphinha)

In [ ]:
scaled, _ = scale_features(scored, scaler=scaler, fit=False)
row = scored.loc[scored['name'] == 'Raphinha'].iloc[0]
contrib = feature_contributions(scaled.loc[row.name], row['profile'], weights)
contrib.head(8)

## 4. Surrogate vs expert (holdout)

In [ ]:
bundle = PositionModelBundle.load(ROOT / 'models')
compare_expert_vs_surrogate(match, weights, bundle)